# Imports

In [1]:
import pandas as pd
import networkx as nx
import re

# Load data

In [2]:
DATA_PATH = '../Data/test_KTH_only.json'
SAVE_GRAPH_PATH = 'graph/'

In [3]:
df = pd.read_json(DATA_PATH)
df.head()

,PID,Title,Keywords,Content,Authors
0,680998,2D Mapping Solutionsfor Low Cost Mobile Robot,[],"\nMapping, localization, and path-planning are...","[WANG, XUAN]"
1,1903661,3D Scene Reconstruction using Diffusion Models...,"[Computer Vision, 3D Reconstruction, Text-to-3...",\nState of the art text-to-image diffusion mod...,"[Persson, Carl]"
2,1229779,3D Shape Detection for Augmented Reality (3D f...,"[3D Machine Learning, object detection, comput...","\nIn previous work, 2D object recognition has ...","[Anadon Leon, Hector]"
3,1888229,3D Shape Retrieval Meets Machine Learning : Im...,"[3D shape retrieval, pattern recognition, mach...",\nIn the rapidly evolving realm of CAD model r...,"[Hazara, Omid]"
4,1245296,3D YOLO: End-to-End 3D Object Detection Using ...,"[Computer vision, Machine Learning, Autonomous...","\nFor safe and reliable driving, it is essenti...","[Al Hakim, Ezeddin]"


# Build graph

In [4]:
ignore = set([
    "DEPARTMENT OF ZOOLOGY, STOCKHOLM UNIVERSITY, STOCKHOLM, SWEDEN.)",
    "JHERONIMUS ACAD DATA SCI, NL-5211 DA SHERTOGENBOSCH, NETHERLANDS.)",
    "DEPARTMENT OF ZOOLOGY, STOCKHOLM UNIVERSITY, STOCKHOLM, SWEDEN)",
    "ENJOYOR CO., LTD, HANGZHOU 310030, CHINA.)",
    "DIGITAL FUTURES, KTH, STOCKHOLM)",
    "MOHAMED BIN ZAYED UNIVERSITY OF ARTIFICIAL INTELLIGENCE)",  
])

In [5]:
def extract_author_names(authors):
    names = []
    for author in authors.split(';'):
        match = re.match(r'^([^\[\(\]]+)', author)
        if match:
            names.append(match.group(1).strip().upper())
    return names

In [6]:
g = nx.Graph()

for _, row in df.iterrows():
    author_names = [name.upper() for name in row.Authors]
    for i in range(0, len(author_names)):
        if author_names[i] in ignore:
            continue
        for j in range(i+1, len(author_names)):
            if author_names[j] in ignore:
                continue
            g.add_edge(author_names[i], author_names[j])
    

# Authors to author ids

In [8]:
author_ids = {name: i for i, name in enumerate(g.nodes())}
author_ids

{'STRÖMBERG, PHILIP': 0,
 'BLOMKVIST KARLSSON, VERA': 1,
 'RENMAN, CASPER': 2,
 'FRISTEDT, HAMPUS': 3,
 'ISHII, SHOTARO': 4,
 'LJUNGGREN, DAVID': 5,
 'BAJALLAN, REBWAR': 6,
 'HASHI, BURHAN': 7,
 'FREDRIKSON, RASMUS': 8,
 'DAHL, JONAS': 9,
 'TASKIN, KASIM': 10,
 'DINLER, MUSTAFA': 11,
 'SALMAN, ALZAHRAA': 12,
 'HANNA, ROUWAYD': 13,
 'ANDERSSON, PETTER': 14,
 'WÖRLUND, ROBERT': 15,
 'HAGLUND, ISAC': 16,
 'JOHANSSON, LISA': 17,
 'ALAM, JOY': 18,
 'LJUNGEHED, JESPER': 19,
 'BESSELING, JOHAN': 20,
 'RENSTRÖM, ANDERS': 21,
 'FORSBERG, TOM-HENRIK': 22,
 'SUNDSTRÖM, JOHAN': 23,
 'LARSSON, MATTIAS': 24,
 'KIRICHENKO, DAN': 25,
 'RANDLEFF, VERONICA': 26,
 'SCHWERMER, PATRIK': 27,
 'STREIJFFERT, NILS': 28,
 'TEGELMARK, FRANS': 29,
 'SUNDLÖF, CLAUDIUS': 30,
 'KRANTZ, GUSTAV': 31,
 'SVEBRANT, HENRIK': 32,
 'SVANBERG, JOHN': 33,
 'GUNÉR, GUSTAF': 34,
 'HAIDER, ADIBBIN': 35,
 'VARATHARAJAH, THUJEEPAN': 36,
 'VICTOR, ERIKSSON': 37,
 'WEDIN, MATTIAS': 38,
 'BENGTSSON, ISAK': 39,
 'RAKSANYI, EMIL': 40,


In [9]:
# Write id-to-author file
with open(SAVE_GRAPH_PATH+'id-to-author.csv', 'w') as file:
    file.write(f'id;name\n')
    for author_name, author_id in author_ids.items():
        file.write(f"{author_id};{author_name}\n")

# Write the n-hop neighbours lists

In [28]:
def n_hop_neighbours(G, node, n):
    '''
    '''
    visited = set([node])
    current_layer = set([node])
    hop_neighbors = []

    for _ in range(n):
        next_layer = set()
        for u in current_layer:
            neighbors = set(G.neighbors(u)) - visited
            next_layer |= neighbors
        hop_neighbors.append(list(next_layer))
        visited |= next_layer
        current_layer = next_layer

    return hop_neighbors

In [31]:
def write_graph(path, n_degrees):
    assert type(n_degrees) == int and n_degrees >= 1, 'n_degrees must be an int >= 1'
    with open(path+f'graph.csv', 'w') as file:
        # Write first row
        first_row = 'node'
        for i in range(1, n_degrees+1):
            first_row += f';{i}_hop_neighbours'
        file.write(first_row + '\n')
        # Write rows
        for node in g.nodes:
            line = f'{author_ids[node]}'
            neighbour_lists = n_hop_neighbours(G=g, node=node, n=n_degrees)
            for neighbour_list in neighbour_lists:
                neighbours_string = [author_ids[name] for name in neighbour_list]
                line += f';{neighbours_string}'
            file.write(line + '\n')

In [32]:
write_graph(path=SAVE_GRAPH_PATH, n_degrees=3)

In [ ]:
df = pd.read_csv('graph/graph.csv', sep=';')

,node,1_hop_neighbours,2_hop_neighbours,3_hop_neighbours
0,0,[1],[],[]
1,1,[0],[],[]
2,2,[3],[],[]
3,3,[2],[],[]
4,4,[5],[],[]
